In [ ]:
# ============================================================
# 📦 Install dependencies
# ============================================================
# Run this cell once, then restart kernel
# !pip install transformers torch accelerate

In [ ]:
# ============================================================
# 🧠 Nibras — LLM Word Completion (Qwen2-0.5B, CPU)
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"   # lightweight, strong, CPU-friendly
AVAILABLE_LETTERS = set('ETAOINSR')        # the 8 BCI letters

# ------------------------------------------------------------
# Load model (first run will download ~1GB)
# ------------------------------------------------------------
print("Loading model... (this may take a moment on first run)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,   # float32 for CPU stability
    device_map="cpu"
)
model.eval()
print("Model loaded.\n")

In [ ]:
# ============================================================
# 🔤 Few-Shot Prompt + Inference
# ============================================================

def build_prompt(user_input: str) -> str:
    """
    Build a few-shot prompt that instructs the model to predict
    the most likely intended English word from a constrained
    letter sequence (may be incomplete or contain substitutions).
    """
    system_message = (
        "You are a spelling correction and word completion assistant. "
        "The user is typing words using only 8 letters: E, T, A, O, I, N, S, R. "
        "Some letters in the target word may be missing or substituted with the nearest available letter. "
        "Your task is to return ONLY the single most likely intended English word. "
        "Do not explain. Do not return a sentence. Return one word only."
    )

    few_shot_examples = [
        {"input": "RANI",   "output": "RAIN"},
        {"input": "TRAI",   "output": "TRAIN"},
        {"input": "STARE",  "output": "STARE"},
        {"input": "OTION",  "output": "MOTION"},
        {"input": "NITE",   "output": "NIGHT"},
        {"input": "EATIN",  "output": "EATING"},
        {"input": "INSTR",  "output": "INSTR"},
    ]

    # Build chat messages
    messages = [{"role": "system", "content": system_message}]

    for ex in few_shot_examples:
        messages.append({"role": "user",      "content": ex["input"]})
        messages.append({"role": "assistant", "content": ex["output"]})

    # Add the actual query
    messages.append({"role": "user", "content": user_input.upper()})

    return messages


def predict_word(user_input: str, max_new_tokens: int = 10) -> str:
    """
    Takes an approximate letter sequence and returns the
    single most likely intended word.
    """
    messages = build_prompt(user_input)

    # Apply chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,        # greedy — deterministic, faster on CPU
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens
    generated = outputs[0][inputs.input_ids.shape[-1]:]
    result = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # Extract first word only in case model returns extra text
    first_word = re.split(r'[\s\.,;!?]', result)[0].strip()
    return first_word

In [ ]:
# ============================================================
# 🧪 Interactive Testing
# ============================================================

test_inputs = [
    "RANI",
    "TRAI",
    "OTION",
    "NITE",
    "EATIN",
    "STORI",
    "EART",
    "TRIAN",
]

print(f"{'Input':<15} {'Predicted Word'}")
print('-' * 35)

for inp in test_inputs:
    prediction = predict_word(inp)
    print(f"{inp:<15} {prediction}")

In [ ]:
# ============================================================
# ⌨️ Manual Input Mode
# ============================================================

print("Enter a letter sequence using E, T, A, O, I, N, S, R.")
print("Type 'quit' to exit.\n")

while True:
    user_input = input("Input: ").strip()
    if user_input.lower() == 'quit':
        break
    if not user_input:
        continue
    result = predict_word(user_input)
    print(f"Predicted word: {result}\n")